# Entrenamiento de YOLO OBB (Oriented Bounding Boxes)

### 1. Instalación de Dependencias
Instalamos la librería de Ultralytics (YOLO) y otras dependencias necesarias.

In [1]:
# Instalar Ultralytics
%pip install ultralytics roboflow matplotlib opencv-python numpy pyyaml

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 95.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 144.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 104.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 197.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 236.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.4/833.4 kB 132.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 235.1 MB/s  0:00:00m0:00:01
  Attempting uninstall: idna0m╸━━━━━━━━━━━━━━━━━━━━━  7/15 [opencv-python]headless]
    Found existing installation: idna 3.18━━━━━━━━━━━━━━━━━━━━  7/15 [opencv-python]
    Uninstalling idna-3.18:╸━━━━━━━━━━━━━━━━━━━━━  7/15 [opencv-python]
      Successfully uninstalled idna-3.18m━━━━━━

### 2. Verificar y corregir rutas en `data.yaml`
Ejecutamos este bloque para asegurarnos de que las rutas relativas sean correctas.

In [2]:
import yaml
import os

yaml_path = "../data/dataset_obb/data.yaml"
EXTRACT_DIR = "../data/dataset_obb/"

if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    data['path'] = os.path.abspath(EXTRACT_DIR)
    data['train'] = 'train/images'
    data['val'] = 'valid/images'
    if 'test' in data:
        data['test'] = 'test/images'

    with open(yaml_path, 'w') as f:
        yaml.safe_dump(data, f, default_flow_style=False)

    print("Archivo data.yaml actualizado con éxito:")
    with open(yaml_path, 'r') as f:
        print(f.read())
else:
    print("ERROR: No se encontró el archivo data.yaml en la carpeta especificada.")

Archivo data.yaml actualizado con éxito:
names:
- TT Net
- TT Racket
- TT Table
nc: 3
path: /teamspace/studios/this_studio/table-tennis-cv/data/dataset_obb
roboflow:
  license: CC BY 4.0
  project: table-tennis-tgntu
  url: https://universe.roboflow.com/agustina-hourcade/table-tennis-tgntu/dataset/2
  version: 2
  workspace: agustina-hourcade
test: test/images
train: train/images
val: valid/images



### 3. Configurar y Entrenar el Modelo YOLO OBB


In [3]:
from ultralytics import YOLO

# 1. Cargar el modelo OBB preentrenado
model = YOLO('yolo26m-obb.pt')

# 2. Entrenar el modelo
results = model.train(
    data      = yaml_path,
    epochs    = 300,
    patience  = 50,
    imgsz     = 960,
    batch     = 16,
    optimizer = 'AdamW',
    lr0       = 0.001,

    # Augmentation
    mosaic    = 1.0,
    mixup     = 0.1,
    degrees   = 30.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,

    # Save configuration
    project   = '../runs',
    name      = 'tenis_mesa_obb_run',
    exist_ok  = True,
    save      = True,
    plots     = True
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/teamspace/studios/this_studio/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.75 🚀 Python-3.12.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/dataset_obb/data.yaml, degrees=30.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.